# ETL on labeled head dataset

I pre-labeled the head dataset using the author's data analysis script and regex. Afterwards, I uploaded the data to Label Studio, to double check the pre-assigned labels. I corrected them where necessary. 

In the following, I am loading the labeled dataset and transforming it to a useful format. I also †ransform the data to be more useable like removing duplicates or replacing string representations of digits to digits ("three" - 3)

In [1]:
import re

import pandas as pd

## Load data

In [2]:
annotations = pd.read_json("temp_head_labeling_label_studio_2025-02-11-17-05-e95fdb59.json", orient="records")
annotations.head()

,id,annotations,file_upload,drafts,predictions,data,meta,created_at,updated_at,inner_id,total_annotations,cancelled_annotations,total_predictions,comment_count,unresolved_comment_count,last_comment_updated_at,project,updated_by,comment_authors
0,4702,"[{'id': 17, 'completed_by': 1, 'result': [{'va...",17085618-temp_head_labeling_bool_as_str.csv,[],[],{'question': 'How many years did Art Carney as...,{},2025-02-11 11:45:36.869253+00:00,2025-02-11 11:46:20.843612+00:00,1,1,0,0,0,0,NaT,3,1,[]
1,4703,"[{'id': 18, 'completed_by': 1, 'result': [{'va...",17085618-temp_head_labeling_bool_as_str.csv,[],[],{'question': 'How many total years was Art Car...,{},2025-02-11 11:45:36.869297+00:00,2025-02-11 11:46:24.061761+00:00,2,1,0,0,0,0,NaT,3,1,[]
2,4704,"[{'id': 19, 'completed_by': 1, 'result': [], '...",17085618-temp_head_labeling_bool_as_str.csv,[],[],{'question': 'Which spouse was Art Carney marr...,{},2025-02-11 11:45:36.869325+00:00,2025-02-11 11:46:34.905627+00:00,3,0,1,0,0,0,NaT,3,1,[]
3,4705,"[{'id': 20, 'completed_by': 1, 'result': [{'va...",17085618-temp_head_labeling_bool_as_str.csv,[],[],{'question': 'How many years before he died wa...,{},2025-02-11 11:45:36.869349+00:00,2025-02-11 11:46:54.025841+00:00,4,1,0,0,0,0,NaT,3,1,[]
4,4706,"[{'id': 21, 'completed_by': 1, 'result': [{'va...",17085618-temp_head_labeling_bool_as_str.csv,[],[],{'question': 'How old was Art Carney when he f...,{},2025-02-11 11:45:36.869376+00:00,2025-02-11 11:46:56.820767+00:00,5,1,0,0,0,0,NaT,3,1,[]


In [3]:
annotations = annotations.loc[:, ["annotations", "data"]]

## Unnest JSON in with labels and original data

In [4]:
annotations["data"].iloc[0]

{'question': 'How many years did Art Carney as actor since 1939?',
 'answer': '54 Years',
 'table_id': 2,
 'answer_type': 'TEMPORAL',
 'is_temporal': 'Is temporal'}

In [5]:
# Add key to avoid gotcha from lambda that causes late-binding closure 
# where the key variable is bound at execution not at definition time. 
data_columns = list(annotations["data"].iloc[0].keys())
annotations = annotations.assign(
    **{key: lambda x, k=key: x["data"].apply(lambda y: y[k]) for key in data_columns}
).drop(columns="data")
annotations.head()

,annotations,question,answer,table_id,answer_type,is_temporal
0,"[{'id': 17, 'completed_by': 1, 'result': [{'va...",How many years did Art Carney as actor since 1...,54 Years,2,TEMPORAL,Is temporal
1,"[{'id': 18, 'completed_by': 1, 'result': [{'va...",How many total years was Art Carney married to...,28 years,2,TEMPORAL,Is temporal
2,"[{'id': 19, 'completed_by': 1, 'result': [], '...",Which spouse was Art Carney married to the least?,Barbara Isaac,2,UNKNOWN,Not temporal
3,"[{'id': 20, 'completed_by': 1, 'result': [{'va...",How many years before he died was Art Carney m...,23,2,COUNT,Is temporal
4,"[{'id': 21, 'completed_by': 1, 'result': [{'va...",How old was Art Carney when he first got divor...,47,2,AGE,Is temporal


In [6]:
# Investigate deeply nested labeles 
annotations["annotations"].iloc[0][0]["result"][0]["value"]["choices"][0]

'true'

In [7]:
# Some data was not labeled because `is_temporal` == "Not temporal". 
# This data was already checked before (in Excel)
# Therefore, I need extract the labels only if they exists
annotations = annotations.assign(annotations=lambda x: x["annotations"].apply(lambda y: y[0]["result"]))

annotations = annotations.assign(
    temporal_class_correct=lambda x: x["annotations"].apply(
        lambda y: y[0]["value"]["choices"][0] if len(y) else y
    )
).drop(columns="annotations")
annotations.head()

,question,answer,table_id,answer_type,is_temporal,temporal_class_correct
0,How many years did Art Carney as actor since 1...,54 Years,2,TEMPORAL,Is temporal,true
1,How many total years was Art Carney married to...,28 years,2,TEMPORAL,Is temporal,true
2,Which spouse was Art Carney married to the least?,Barbara Isaac,2,UNKNOWN,Not temporal,[]
3,How many years before he died was Art Carney m...,23,2,COUNT,Is temporal,true
4,How old was Art Carney when he first got divor...,47,2,AGE,Is temporal,true


In [8]:
# Data where no label was given
annotations.loc[annotations.loc[:, "temporal_class_correct"].str.len()==0, "temporal_class_correct"]

2       []
1058    []
Name: temporal_class_correct, dtype: object

In [9]:
# Replace labels that are [] with "true"
annotations.loc[annotations.loc[:, "temporal_class_correct"].str.len()==0, "temporal_class_correct"] = "true"

# All pre-labeled questions that are wrong are "Is temporal". 
# Reason being that I already had corrected the "Not temporal" in Excel manually.
annotations.loc[annotations.loc[:, "temporal_class_correct"] == "false", "is_temporal"].unique()

array(['Is temporal'], dtype=object)

In [10]:
# Change pre-assigned label if it is wrong
annotations.loc[annotations.loc[:, "temporal_class_correct"] == "false", "is_temporal"] = "Not temporal"
# Keep only temporal QA-pairs
annotations = annotations.drop(columns="temporal_class_correct").query("is_temporal == 'Is temporal'")

## Data cleaning

In [11]:
annotations = annotations.drop_duplicates(subset="question")

In [12]:
word_to_digit = {
    "zero": "0",
    "one": "1",
    "two": "2",
    "three": "3",
    "four": "4",
    "five": "5",
    "six": "6",
    "seven": "7",
    "eight": "8",
    "nine": "9",
    "ten": "10",
}

pipe_between_words = "|".join(word_to_digit.keys())
pattern = re.compile(r"\b(" + pipe_between_words + r")\b")


def replace_num_word(match):
    word = match.group(1)  # The matched word (e.g. "one")
    return word_to_digit[word]  # The corresponding digit (e.g. "1")

In [13]:
annotations = annotations.assign(
    answer_old=lambda x: x["answer"],
    answer=lambda x: x["answer_old"].apply(lambda y: pattern.sub(replace_num_word, y))
)
annotations.query("answer!=answer_old")

,question,answer,table_id,answer_type,is_temporal,answer_old
25,How long did each of Joan Crawford's marriages...,4,8,UNKNOWN,Is temporal,four
192,How many years after Rudolf Thienel became Pre...,0,254,COUNT,Is temporal,zero
252,How many years was Hulk Hogan a wrestler when ...,5,339,COUNT,Is temporal,five
258,How many months after being apprehended did Je...,1,340,COUNT,Is temporal,one
396,How many months after the Plan of Iguala did t...,7,433,COUNT,Is temporal,seven
485,How many years did Davy Jones compete in the L...,4,517,COUNT,Is temporal,four
662,How many years apart were there when Claire Le...,1,671,COUNT,Is temporal,one
684,How many months ago was Aruna Quadri's highest...,1,710,COUNT,Is temporal,one
865,How many years was Theo Bot in the Reserve bef...,4,915,COUNT,Is temporal,four
878,How long after Dick King-Smith's first wife d...,1 year,929,TEMPORAL,Is temporal,one year


## Identify answers with unclear timeunti

In [14]:
# Questions without time units and answer without one too
annotations.query(
    "answer.str.match('^\d+$') " 
    "and ~question.str.lower().str.contains('year') "
    "and ~question.str.lower().str.contains('month') "
    "and ~question.str.lower().str.contains('day') "
    "and ~question.str.lower().str.contains('age') " 
    "and ~question.str.lower().str.contains('old') "
    "and ~answer.str.match('\d{4}')"
).question.values

array(['How long had James Worthy been playing professionally when he won the NCAA Championship?',
       "How long was it between when Bayinnaung's reign as Suzerain of Lan Na began and his first reign as Suzerain of Lan Xang ended?",
       'Chrisann Brennan and Steven Paul jobs are Partner for how long?',
       'How long ago did Faith die?',
       'How long was Robert K. Silverstein a producer for Access Hollywood?',
       'How long after production stopped on the Firefly was it officially retired?',
       'How long after the First Flight was the Firefly retired?',
       'How long after the Firefly was introduced did production stop?'],
      dtype=object)

In [15]:
annotations.query("question=='How long ago did Faith die?'")

,question,answer,table_id,answer_type,is_temporal,answer_old
446,How long ago did Faith die?,46,472,UNKNOWN,Is temporal,46


In [16]:
annotations.to_csv("labeled_cleaned_head_dataset.csv", index=False)